# Plan 10: Fisher-speed feasibility and direct response

This notebook reads the immutable Phase 1--1d artifacts. It first asks whether the retained population path is smooth enough to support the fixed-composition pace inversion, then tests the finite-EWC decision consequence directly without a population-reference path.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mnist_experiment.rotated_mnist.pace_control.config import load_feasibility_config
from mnist_experiment.rotated_mnist.pace_control.finite_risk_config import load_finite_risk_config
from mnist_experiment.rotated_mnist.pace_control.reference_config import load_reference_repair_config
from mnist_experiment.rotated_mnist.pace_control.response_surface_config import load_response_surface_config

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mnist_experiment/plan10.md').is_file()), None)
if root is None:
    raise RuntimeError('Run the notebook from within the repository tree')
config = load_feasibility_config(root / 'mnist_experiment/rotated_mnist/pace_control/configs/phase1_feasibility.json')
run = root / 'cache/mnist_experiment/rotated_mnist/plan10/phase1' / config.run_id
if not (run / 'COMPLETED').is_file():
    raise RuntimeError(f'Phase 1 artifact is incomplete or absent: {run}')
load = lambda name: json.loads((run / name).read_text())
summary = load('summary.json')
speed = pd.DataFrame(load('fisher_speed_map.json'))
validation = pd.DataFrame(load('finite_step_validation.json'))
route = pd.DataFrame(load('oracle_route.json'))
finite_config = load_finite_risk_config(root / 'mnist_experiment/rotated_mnist/pace_control/configs/phase1b_finite_risk.json')
finite_run = root / 'cache/mnist_experiment/rotated_mnist/plan10/phase1b' / finite_config.run_id
if not (finite_run / 'COMPLETED').is_file():
    raise RuntimeError(f'Phase 1b artifact is incomplete or absent: {finite_run}')
load_finite = lambda name: json.loads((finite_run / name).read_text())
finite_summary = load_finite('summary.json')
pairs = pd.DataFrame(load_finite('finite_risk_pairs.json'))
finite_route = pd.DataFrame(load_finite('oracle_route.json'))
panel = pd.DataFrame(load_finite('reference_panel.json'))
repair_config = load_reference_repair_config(root / 'mnist_experiment/rotated_mnist/pace_control/configs/phase1c_reference_repair.json')
repair_run = root / 'cache/mnist_experiment/rotated_mnist/plan10/phase1c/screen' / repair_config.run_id
if not (repair_run / 'COMPLETED').is_file():
    raise RuntimeError(f'Phase 1c artifact is incomplete or absent: {repair_run}')
load_repair = lambda name: json.loads((repair_run / name).read_text())
repair_summary = load_repair('summary.json')
repair_panel = pd.DataFrame(load_repair('heldout_panel.json'))
repair_pairs = pd.DataFrame(load_repair('finite_risk_pairs.json'))
repair_fits = pd.DataFrame(load_repair('fit_metrics.json'))
response_config = load_response_surface_config(root / 'mnist_experiment/rotated_mnist/pace_control/configs/phase1d_full.json')
response_run = root / 'cache/mnist_experiment/rotated_mnist/plan10/phase1d' / response_config.run_id
if not (response_run / 'COMPLETED').is_file():
    raise RuntimeError(f'Phase 1d artifact is incomplete or absent: {response_run}')
load_response = lambda name: json.loads((response_run / name).read_text())
response_summary = load_response('summary.json')
response_surface = pd.DataFrame(load_response('response_surface.json'))
response_gate = load_response('gate.json')

## Gate result

The gate requires nonnegative targets, locally monotone finite Fisher energy, a useful quadratic approximation, route completion, and less than 50% pace-bound occupancy. A failed gate stops the one-step EWC pilot; it is not repaired by selecting new bounds after seeing the outcome.

In [ ]:
pd.DataFrame({
    'value': {
        'Gate passed': summary['gate_pass'],
        'Map angles': summary['map_angle_count'],
        'Median quadratic relative error': summary['median_relative_approximation_error'],
        'Monotone larger-step fraction': summary['monotone_fraction'],
        'Pace-bound fraction': summary['bound_fraction'],
        'Infeasible route steps': summary['infeasible_steps'],
        'Route observations': summary['route_observations'],
    }
})

## Local Fisher speed

With $\varphi$ measured in degrees, the local target is $J(\varphi)=(d\theta^\star/d\varphi)^T\mathsf M(\varphi)(d\theta^\star/d\varphi)$. Paired local-MLE clouds remove estimated finite-reference noise. Exact zeros below occur where neighboring retained population parameter vectors are identical after fitting.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(speed['angle_degrees'], speed['fisher_speed'], color='#136f63', marker='o', ms=3, lw=1.6)
zeros = speed[speed['fisher_speed'] == 0]
ax.scatter(zeros['angle_degrees'], zeros['fisher_speed'], color='#ba3b46', s=30, zorder=3, label='Debiased speed is zero')
ax.set(xlabel='Rotation angle (degrees)', ylabel='Fisher energy / degree$^2$', title='Retained population path is locally discontinuous or flat')
ax.grid(alpha=.2)
ax.legend(frameon=False, loc='upper right')
plt.tight_layout()

## Finite-step check

A usable local map should satisfy $\|\theta^\star(\varphi+\delta)-\theta^\star(\varphi)\|_{\mathsf M}^2\approx J(\varphi)\delta^2$ and increase with $|\delta|$. The diagonal is exact agreement.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for increment, group in validation.groupby('increment_degrees'):
    axes[0].scatter(group['quadratic_prediction'], group['energy'], s=16, alpha=.65, label=rf'$\delta={increment:g}^\circ$')
limit = max(validation['quadratic_prediction'].max(), validation['energy'].max())
axes[0].plot([0, limit], [0, limit], color='black', lw=1, ls='--')
axes[0].set(xlabel='Local quadratic prediction', ylabel='Observed finite Fisher energy', title='Quadratic fidelity')
axes[0].legend(frameon=False)
pace_colors = route['pace_bound'].fillna('interior').map({'lower': '#ba3b46', 'upper': '#d17b0f', 'interior': '#136f63'})
axes[1].scatter(route['step'], route['pace_degrees'], c=pace_colors, s=14)
axes[1].axhline(config.minimum_pace_degrees, color='#ba3b46', lw=1, ls=':')
axes[1].axhline(config.maximum_pace_degrees, color='#d17b0f', lw=1, ls=':')
axes[1].set(xlabel='Accepted update', ylabel='Angular pace (degrees)', title='Inverted route is almost entirely clipped')
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout()

## Derivative-map decision

Phase 1 is rejected under its frozen gate. The retained 10,000-observation reference fits do not identify a smooth local Euclidean optimum path: 9 of 41 speed estimates are zero, the finite-step energy check is nonmonotone too often, and the implied route uses a pace bound on nearly every update. This does not falsify the algebraic inversion; it rejects parameter differencing as the local pace measurement on these retained references.

## Phase 1b: finite excess risk

The recovery measures movement without subtracting parameters. For a target angle $\psi=\varphi+\delta$,

$$
S_{\mathrm{NLL}}(\varphi,\delta)=2[\mathcal L_{\psi}(\theta^\star(\varphi))-\mathcal L_{\psi}(\theta^\star(\psi))].
$$

This equals the local Fisher quadratic to second order at an interior optimum. Both losses use the same 8,000 observations excluded from checkpoint selection; negative values remain visible.

In [ ]:
pd.DataFrame({'value': {
    'Gate passed': finite_summary['gate_pass'],
    'Nonnegative energy fraction': finite_summary['nonnegative_energy_fraction'],
    'Nested monotone fraction': finite_summary['nested_monotone_fraction'],
    'Global Fisher-energy Spearman': finite_summary['fisher_energy_spearman'],
    r'Local Fisher-energy Spearman ($|\delta|\leq3^\circ$)': finite_summary['local_fisher_energy_spearman'],
    'Route target attainment': finite_summary['route_target_attainment_fraction'],
    'Median route target error': finite_summary['median_route_target_relative_error'],
    'Nominal references best on own held-out panel': f"{int(panel['own_reference_is_best'].sum())}/{len(panel)}",
}})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
distance = pairs.groupby('increment_degrees')['finite_excess_nll_energy'].agg(['median', lambda x: x.quantile(.1), lambda x: x.quantile(.9)]).reset_index()
axes[0].fill_between(distance['increment_degrees'], distance['<lambda_0>'], distance['<lambda_1>'], color='#6aa5a0', alpha=.25, label='10th-90th percentile')
axes[0].plot(distance['increment_degrees'], distance['median'], color='#136f63', lw=2, label='Median')
axes[0].axhline(0, color='black', lw=1, ls='--')
axes[0].set(xlabel='Angular increment (degrees)', ylabel='Finite excess-NLL energy', title='Globally orderly, locally unresolved')
axes[0].legend(frameon=False)
positive = pairs[pairs['fisher_energy'] > 0]
axes[1].scatter(positive['fisher_energy'], positive['finite_excess_nll_energy'], c=positive['increment_degrees'], cmap='viridis', s=10, alpha=.55)
axes[1].axhline(0, color='black', lw=1, ls='--')
axes[1].set(xlabel='Debiased Fisher quadratic', ylabel='Finite excess-NLL energy', title='Agreement appears only over larger moves')
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(finite_route['step'], finite_route['movement_target'], marker='o', lw=1.8, label=r'Required $S_t^\dagger$')
ax.plot(finite_route['step'], finite_route['finite_excess_nll_energy'], marker='s', lw=1.5, label='Selected finite energy')
misses = finite_route[~finite_route['target_attained']]
ax.scatter(misses['step'], misses['finite_excess_nll_energy'], color='#ba3b46', s=45, zorder=3, label='Target missed')
ax.set(xlabel='Route update', ylabel='Movement energy', title='Finite-risk route target fidelity')
ax.grid(alpha=.2)
ax.legend(frameon=False, ncol=3, loc='upper center', bbox_to_anchor=(.5, 1.18))
plt.tight_layout()

## Recovery decision

Phase 1b also rejects its frozen gate. The finite-risk metric is a useful global diagnostic, but it exposes a reference-estimation failure: only 2 of 41 nominal target checkpoints have the lowest held-out NLL among the retained states at their own angle. At increments no larger than $3^\circ$, excess risk is nonnegative only 61.7% of the time and its Fisher-energy rank correlation is 0.088. Only 58.3% of selected route moves attain $S_t^\dagger$. A predeclared multi-start repair tests whether better optimization resolves this failure.

## Phase 1c: multi-start reference repair

Each angle receives three independently refined starts chosen on a 2,000-observation selection panel. Every fit runs 12 Adam epochs on the same 10,000 training observations, and the selected states are then cross-evaluated on a disjoint 8,000-observation held-out panel. The screen is intentionally completed before any new Fisher or local-MLE work.

In [ ]:
pd.DataFrame({'value': {
    'Gate passed': repair_summary['gate_pass'],
    'Candidate fits': repair_summary['candidate_fits'],
    'Median own-angle NLL improvement': repair_summary['median_own_nll_improvement'],
    'Own reference best fraction': repair_summary['own_reference_best_fraction'],
    'Median held-out regret': repair_summary['median_heldout_regret'],
    'All-pair nonnegative fraction': repair_summary['nonnegative_energy_fraction'],
    r'Local nonnegative fraction ($|\delta|\leq3^\circ$)': repair_summary['local_nonnegative_energy_fraction'],
    'Nested monotone fraction': repair_summary['nested_monotone_fraction'],
}})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].plot(repair_panel['angle_degrees'], repair_panel['original_own_nll'], color='#787878', lw=1.7, label='Inherited reference')
axes[0].plot(repair_panel['angle_degrees'], repair_panel['repaired_own_nll'], color='#136f63', lw=2, label='Multi-start repair')
axes[0].set(xlabel='Rotation angle (degrees)', ylabel='Own-angle held-out NLL', title='Refinement materially improves the fits')
axes[0].legend(frameon=False)
axes[1].plot(repair_panel['angle_degrees'], repair_panel['heldout_regret'], color='#ba3b46', lw=1.8)
axes[1].axhline(repair_config.maximum_median_regret, color='black', lw=1, ls='--', label='Median-regret gate')
best = repair_panel[repair_panel['repaired_own_is_best']]
axes[1].scatter(best['angle_degrees'], best['heldout_regret'], color='#136f63', s=28, zorder=3, label='Own state is best')
axes[1].set(xlabel='Rotation angle (degrees)', ylabel='Regret to best repaired state', title='Fine angle identity remains unresolved')
axes[1].legend(frameon=False)
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout()

In [ ]:
distance = repair_pairs.groupby('increment_degrees')['finite_excess_nll_energy'].agg(['median', lambda x: x.quantile(.1), lambda x: x.quantile(.9)]).reset_index()
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.fill_between(distance['increment_degrees'], distance['<lambda_0>'], distance['<lambda_1>'], color='#6aa5a0', alpha=.25, label='10th-90th percentile')
ax.plot(distance['increment_degrees'], distance['median'], color='#136f63', lw=2, label='Median')
ax.axhline(0, color='black', lw=1, ls='--')
ax.axvspan(0, repair_config.local_increment_limit_degrees, color='#d17b0f', alpha=.08, label='Frozen local region')
ax.set(xlabel='Angular increment (degrees)', ylabel='Finite excess-NLL energy', title='Global order improves; local signs do not')
ax.grid(alpha=.2)
ax.legend(frameon=False)
plt.tight_layout()

## Phase 1d: direct finite-EWC response surface

Phase 1c materially improves own-angle losses, but only 4 of 41 repaired states are best at their nominal held-out angle. Phase 1d therefore bypasses the disputed population-reference path. At four frozen historical anchors, each pace and four-observation batch is paired across seven diagnostic composition actions. The gate asks whether the empirical finite-EWC optimum moves through $\bar\pi=.05$ within the original `.375`--`1.5` degree pace bounds.

In [ ]:
in_bound = response_surface[response_surface['within_physical_bounds']].copy()
in_bound['point_regret'] = in_bound['regret'].map(lambda value: value['point_regret'])
in_bound['upper_regret'] = in_bound['regret'].map(lambda value: value['upper_regret'])
pd.DataFrame({'value': {
    'Gate passed': response_summary['gate_pass'],
    'Finite-EWC fits': response_summary['expected_fit_count'],
    'Failed fits': response_summary['failed_fit_count'],
    'Anchors with point-competitive .05': response_summary['point_competitive_anchor_count'],
    'Anchors with confidence-competitive .05': response_summary['upper_competitive_anchor_count'],
    'Interior confidence-competitive anchors': response_summary['interior_upper_competitive_anchor_count'],
    'Tie-aware monotone fraction': response_summary['monotone_fraction'],
    'Resolved crossing in both directions': response_summary['resolved_crossing_both_directions'],
    'Expansion recommended': response_summary['expansion_recommended'],
}})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
colors = {20: '#136f63', 40: '#d17b0f', 60: '#ba3b46', 100: '#3a506b'}
for anchor, group in in_bound.groupby('anchor_step'):
    axes[0].plot(group['pace_degrees'], group['empirical_best_pi'], marker='o', lw=1.7, color=colors[anchor], label=f'Anchor {anchor}')
axes[0].axhline(response_config.fixed_pi, color='black', lw=1, ls='--', label=r'$\bar\pi=.05$')
axes[0].set(xlabel='Angular pace (degrees)', ylabel='Empirical minimizing action', title='The optimum remains below the fixed action')
axes[0].legend(frameon=False, ncol=2)
for anchor, group in in_bound.groupby('anchor_step'):
    axes[1].plot(group['pace_degrees'], group['point_regret'], marker='o', lw=1.7, color=colors[anchor], label=f'Anchor {anchor}')
axes[1].axhline(response_config.practical_nll_margin, color='black', lw=1, ls='--', label='Practical margin')
axes[1].set(xlabel='Angular pace (degrees)', ylabel='Point regret of .05 (NLL)', title='Every in-bound cell exceeds the margin')
for ax in axes: ax.grid(alpha=.2)
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5))
for direction, color in [('-1', '#ba3b46'), ('1', '#136f63')]:
    contrast = response_gate['crossing_contrasts'][direction]
    pace = np.asarray(contrast['pace_degrees'])
    mean = np.asarray(contrast['mean'])
    lower = np.asarray(contrast['simultaneous_lower'])
    upper = np.asarray(contrast['simultaneous_upper'])
    label = 'Negative direction' if direction == '-1' else 'Positive direction'
    ax.plot(pace, mean, marker='o', lw=1.8, color=color, label=label)
    ax.fill_between(pace, lower, upper, color=color, alpha=.15)
ax.axhline(0, color='black', lw=1, ls='--')
ax.set(xlabel='Angular pace (degrees)', ylabel=r'$\mathcal{R}(.025)-\mathcal{R}(.075)$', title='No low-to-high composition crossing')
ax.grid(alpha=.2)
ax.legend(frameon=False)
plt.tight_layout()

## Phase 1d decision

The direct finite-EWC screen rejects the oracle mechanism independently of the failed reference geometry. Across 16 in-bound anchor/pace cells, the empirical optimum is `.0125` in 10 cells, `.025` in 5, and `.0375` in 1; it is never `.05` or larger. Point regret for `.05` ranges from 0.0113 to 0.1447 NLL with median 0.0612, and no anchor has a confidence-competitive interior pace. The `.025` versus `.075` contrast never reverses in either direction. All 5,376 fits succeed, so this is not an optimizer-failure artifact. The frozen Phase 1d gate fails and the 128-replicate expansion is not authorized.

## Final decision

The repair lowers own-angle held-out NLL by 0.0221 at the median, but it does not recover the local population path. Phase 1d then removes that ambiguity by evaluating actual finite-EWC predictive risk without reference parameters or a Fisher-speed map. The empirical optimum stays below `.05` throughout the physical pace range and at the `3.0`-degree characterization points. The current inverted optimal-$\pi$ pace strategy is therefore not viable for this Rotated-MNIST learner even as an oracle mechanism. This does not falsify the algebraic inversion under its scalar-affine assumptions. No Fisher/cloud rebuild, 128-replicate expansion, or Phase 2 learner experiment is run.